# Notebook 02b — LLM Validation of Element Extraction

**Project:** Scènes à Faire Computational Framework (v4 rebuild)  
**Purpose:** Validate Notebook 02's spaCy element extraction against Claude Sonnet 4.6 on a stratified 1,500-film sample.

**Why this notebook exists:** spaCy extracts surface-level elements (nouns, verbs, named entities) via part-of-speech tags and dependency parses. It cannot identify abstract narrative tropes ("chosen one," "final girl," "redemption arc," "meet-cute") that require semantic understanding. This notebook runs a frontier LLM on a sample of films, compares the two extraction methods, and produces a validation report for the methods section.

**Cost:** ~$10-15 for 1,500 films via Claude Sonnet 4.6. Cached per-film so re-runs don't re-bill.

**API Key:** Stored as a Colab secret named `ANTHROPIC_API_KEY`. Never hardcoded.

**Prerequisites:**
- Notebook 02 must be complete (`elements_by_film.parquet` and `element_index.parquet` exist)
- Anthropic API key stored in Colab secrets (sidebar → Add secret → name: `SAF_VALIDATION_ANTHROPIC_API_KEY`)

## 1. Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q anthropic pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 30.1 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import hashlib
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import anthropic

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis'

UNION_PATH    = f'{PROJECT_ROOT}/v4_rebuild/01c_union/movies_union.parquet'
ELEMENTS_PATH = f'{PROJECT_ROOT}/v4_rebuild/02_elements/elements_by_film.parquet'
INDEX_PATH    = f'{PROJECT_ROOT}/v4_rebuild/02_elements/element_index.parquet'

OUTPUT_DIR = f'{PROJECT_ROOT}/v4_rebuild/02b_llm_validation'
CACHE_DIR  = f'{OUTPUT_DIR}/llm_cache'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

assert os.path.exists(UNION_PATH), 'Union parquet missing. Run Notebook 01c.'
assert os.path.exists(ELEMENTS_PATH), 'Elements parquet missing. Run Notebook 02.'

# Load API key from Colab secrets (never hardcoded)
API_KEY = userdata.get('SAF_VALIDATION_ANTHROPIC_API_KEY')
assert API_KEY and API_KEY.startswith('sk-'), 'API key not found. Add ANTHROPIC_API_KEY to Colab secrets (🔑 sidebar).'

client = anthropic.Anthropic(api_key=API_KEY)
print('✓ Anthropic client initialized')
print(f'Output dir: {OUTPUT_DIR}')

✓ Anthropic client initialized
Output dir: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02b_llm_validation


## 2. Build Stratified Sample

We sample 1,500 films stratified by genre so every canonical genre is represented. For genres with fewer than 100 films we take proportionally fewer. The sample is deterministic (fixed random seed) so re-running produces the same films.

In [ ]:
SAMPLE_SIZE = 1500
RANDOM_SEED = 20260219  # Matching your preregistration seed

union = pd.read_parquet(UNION_PATH)
elements = pd.read_parquet(ELEMENTS_PATH)

print(f'Union corpus:  {len(union):,} films')
print(f'Elements file: {len(elements):,} films')

# Merge to get plot text + extracted elements in one frame
merged = union.merge(elements, on='union_id', how='inner')
print(f'Merged:        {len(merged):,} films')

Union corpus:  45,958 films
Elements file: 45,958 films
Merged:        45,958 films


In [ ]:
CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western'
]

# --- Stratification mode --------------------------------------------------
# 'equal'        : balanced allocation across genres (recommended for a
#                  validation sample - gives every genre a usable N so the
#                  per-genre agreement numbers and the overall mean are not
#                  dominated by whichever genre is largest).
# 'proportional' : allocation mirrors corpus genre prevalence.
# Both are deterministic and seed-stable; the choice is explicit and
# preregisterable. Flip this one line to switch.
STRATIFICATION = 'equal'

# --- primary_genre: computed ONCE, robustly -------------------------------
# genres_canonical comes back from parquet as a numpy array, not a list, so
# isinstance(gs, list) is False. Guard for None, strings, and empties too.
def first_genre(gs):
    if gs is None or isinstance(gs, str):
        return 'Unknown'
    try:
        if len(gs) > 0:
            return gs[0]
    except TypeError:
        return 'Unknown'
    return 'Unknown'

merged['primary_genre'] = merged['genres_canonical'].apply(first_genre)

n_unknown = (merged['primary_genre'] == 'Unknown').sum()
print(f'primary_genre assigned. Unknown: {n_unknown:,} of {len(merged):,} films')

# --- availability per genre ----------------------------------------------
genre_counts = merged['primary_genre'].value_counts()
avail = {g: int(genre_counts.get(g, 0)) for g in CANONICAL_GENRES}

# --- allocation via largest-remainder ------------------------------------
def allocate(sample_size, avail, mode):
    """Allocate sample_size across genres, capped at availability.
    Largest-remainder method: floor each target, then hand out the leftover
    one at a time to whichever eligible genre is furthest below its target.
    Deterministic (genre name breaks ties)."""
    genres = list(avail.keys())
    if mode == 'proportional':
        total = sum(avail.values())
        target = {g: sample_size * avail[g] / total for g in genres}
    elif mode == 'equal':
        per = sample_size / len(genres)
        target = {g: per for g in genres}
    else:
        raise ValueError(f"STRATIFICATION must be 'equal' or 'proportional', got {mode!r}")

    alloc = {g: min(int(target[g]), avail[g]) for g in genres}
    remaining = sample_size - sum(alloc.values())
    while remaining > 0:
        candidates = sorted(
            [g for g in genres if alloc[g] < avail[g]],
            key=lambda g: (-(target[g] - alloc[g]), g)
        )
        if not candidates:
            break  # corpus genuinely too small to reach sample_size
        alloc[candidates[0]] += 1
        remaining -= 1
    return alloc, remaining

samples_per_genre, shortfall = allocate(SAMPLE_SIZE, avail, STRATIFICATION)

print(f'\nSampling plan  (mode={STRATIFICATION}, target {SAMPLE_SIZE}):')
for g in CANONICAL_GENRES:
    cap = '  (capped at availability)' if samples_per_genre[g] == avail[g] else ''
    print(f'  {g:<18} {samples_per_genre[g]:>4}   of {avail[g]:>6} available{cap}')
print(f'  {"TOTAL":<18} {sum(samples_per_genre.values()):>4}')
if shortfall > 0:
    print(f'  NOTE: {shortfall} films short of target - not enough films across '
          f'genres to fill the sample at this size.')

primary_genre assigned. Unknown: 0 of 45,958 films

Sampling plan  (mode=equal, target 1500):
  Action               89   of   7775 available
  Adventure            89   of   2005 available
  Animation            89   of   2157 available
  Comedy               89   of  11623 available
  Crime                88   of   3063 available
  Drama                88   of  12444 available
  Family               88   of    284 available
  Fantasy              88   of    305 available
  History              88   of    380 available
  Horror               88   of   2497 available
  Music                88   of    432 available
  Mystery              88   of    374 available
  Romance              88   of    774 available
  Science Fiction      88   of    380 available
  Thriller             88   of    727 available
  War                  88   of    215 available
  Western              88   of    523 available
  TOTAL              1500


In [ ]:
# Verify the sampling plan before drawing (primary_genre and the allocation
# are both computed in the cell above; this cell only checks them).
print('Primary genre distribution in the corpus (top 10):')
print(merged['primary_genre'].value_counts().head(10).to_string())

assert all(samples_per_genre[g] <= avail[g] for g in CANONICAL_GENRES), \
    'allocation exceeds availability for some genre'
assert sum(samples_per_genre.values()) <= SAMPLE_SIZE, \
    'allocation overshoots SAMPLE_SIZE'

planned = sum(samples_per_genre.values())
mn = min(samples_per_genre.values())
mx = max(samples_per_genre.values())
print(f'\nAllocation check passed: {planned} films planned, '
      f'per-genre range {mn} to {mx}.')

Primary genre distribution in the corpus (top 10):
primary_genre
Drama        12444
Comedy       11623
Action        7775
Crime         3063
Horror        2497
Animation     2157
Adventure     2005
Romance        774
Thriller       727
Western        523

Allocation check passed: 1500 films planned, per-genre range 88 to 89.


In [ ]:
# Draw the stratified sample
rng = np.random.RandomState(RANDOM_SEED)
sample_frames = []

for genre in CANONICAL_GENRES:
    genre_films = merged[merged['primary_genre'] == genre]
    n = min(samples_per_genre[genre], len(genre_films))
    sampled = genre_films.sample(n=n, random_state=rng)
    sample_frames.append(sampled)

sample = pd.concat(sample_frames, ignore_index=True)
print(f'Final sample: {len(sample):,} films')
print(f'Genres represented: {sample["primary_genre"].nunique()}')

Final sample: 1,500 films
Genres represented: 17


In [ ]:
# Fix parquet round-trip type issue on list columns
# (same bug as the genres_canonical issue in sampling)
HAS_CHUNKS = 'chunks' in elements.columns
if not HAS_CHUNKS:
    print('WARNING: elements_by_film.parquet has no "chunks" column. This file '
          'predates the multi-word chunk extraction - re-run Notebook 02 so '
          'that chunks are validated here.\n')

list_cols = ['nouns', 'verbs', 'entities'] + (['chunks'] if HAS_CHUNKS else [])
for col in list_cols:
    elements[col] = elements[col].apply(
        lambda x: list(x) if hasattr(x, '__len__') and not isinstance(x, str) else []
    )

# Verify
print('Column types after fix:')
for col in list_cols:
    val = elements.iloc[0][col]
    print(f'  {col}[0]: {type(val)} — {val[:3]}')

Column types after fix:
  nouns[0]: <class 'list'> — ['nation', 'panem', 'capitol']
  verbs[0]: <class 'list'> — ['consist', 'provide', 'select']
  entities[0]: <class 'list'> — [{'label': 'ORG', 'text': 'panem'}, {'label': 'ORG', 'text': 'capitol'}, {'label': 'EVENT', 'text': 'hunger games'}]
  chunks[0]: <class 'list'> — ['wealthy capitol', 'poorer district', 'past rebellion']


## 3. LLM Extraction Prompt

We ask Claude Sonnet 4.6 to extract narrative elements in structured JSON. The prompt is designed to capture both surface elements (that spaCy catches) and abstract tropes (that spaCy misses), so we can measure the gap.

In [ ]:
EXTRACTION_PROMPT = """You are analyzing a movie plot summary to extract narrative elements for a computational study of genre conventions.

Given the plot summary below, extract ALL narrative elements that could help identify this film's genre. Return ONLY valid JSON with no other text.

Extract these categories:

1. **settings**: Physical locations and environments (e.g., "haunted house", "space station", "frontier town", "hospital", "courtroom")
2. **characters**: Character TYPES and ROLES, not proper names (e.g., "sheriff", "detective", "mad scientist", "damsel in distress", "mentor figure", "final girl")
3. **objects**: Significant objects, weapons, tools, or props (e.g., "sword", "spaceship", "magic ring", "gun", "love letter")
4. **events**: Key plot events, actions, and narrative beats (e.g., "murder investigation", "car chase", "wedding", "alien invasion", "prison escape", "betrayal")
5. **themes**: Abstract narrative themes and tropes (e.g., "redemption", "forbidden love", "revenge", "coming of age", "good vs evil", "survival")

Rules:
- Use lowercase for all elements
- Be specific: "haunted house" is better than "house"; "corrupt sheriff" is better than "sheriff"
- Include BOTH concrete elements AND abstract tropes/themes
- Extract what is PRESENT in the plot, not what you assume about the genre
- Multi-word elements are fine and encouraged ("car chase", "love triangle", "final showdown")
- Aim for 5-15 elements per category, fewer if the plot is short

Return format (JSON only, no markdown fences):
{"settings": [...], "characters": [...], "objects": [...], "events": [...], "themes": [...]}

PLOT SUMMARY:
"""

print('Prompt length:', len(EXTRACTION_PROMPT), 'chars')
print('Preview (last 200 chars):', EXTRACTION_PROMPT[-200:])

Prompt length: 1621 chars
Preview (last 200 chars): ts per category, fewer if the plot is short

Return format (JSON only, no markdown fences):
{"settings": [...], "characters": [...], "objects": [...], "events": [...], "themes": [...]}

PLOT SUMMARY:



## 4. Run LLM Extraction with Caching

For each film in the sample, send the plot to Claude Sonnet 4.6 and parse the JSON response. Each response is cached to a file keyed by `union_id`, so re-running doesn't re-bill. Rate limiting: 0.5 second delay between calls to stay within Anthropic's rate limits.

In [ ]:
MODEL = 'claude-sonnet-4-6'
MAX_PLOT_CHARS = 6000  # Truncate very long plots to control cost
DELAY_SECONDS = 0.5    # Rate limiting between API calls

def get_cache_path(union_id):
    return f'{CACHE_DIR}/{union_id}.json'

def extract_with_llm(plot_text, union_id):
    """Send a single plot to Claude Sonnet and return parsed JSON."""
    # Check cache first
    cache_path = get_cache_path(union_id)
    if os.path.exists(cache_path):
        with open(cache_path, 'r') as f:
            return json.load(f)

    # Truncate plot if very long
    plot_truncated = plot_text[:MAX_PLOT_CHARS]

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            messages=[{
                'role': 'user',
                'content': EXTRACTION_PROMPT + plot_truncated
            }]
        )

        raw_text = response.content[0].text.strip()

        # Strip markdown fences if present
        if raw_text.startswith('```'):
            raw_text = raw_text.split('\n', 1)[-1]
            if raw_text.endswith('```'):
                raw_text = raw_text[:-3].strip()

        parsed = json.loads(raw_text)

        # Validate structure
        result = {
            'settings':   parsed.get('settings', []),
            'characters': parsed.get('characters', []),
            'objects':    parsed.get('objects', []),
            'events':     parsed.get('events', []),
            'themes':     parsed.get('themes', []),
            'raw_text':   raw_text,
            'error':      None
        }

    except json.JSONDecodeError as e:
        result = {
            'settings': [], 'characters': [], 'objects': [],
            'events': [], 'themes': [],
            'raw_text': raw_text if 'raw_text' in dir() else '',
            'error': f'JSON parse error: {str(e)}'
        }
    except Exception as e:
        result = {
            'settings': [], 'characters': [], 'objects': [],
            'events': [], 'themes': [],
            'raw_text': '',
            'error': f'API error: {str(e)}'
        }

    # Cache result
    with open(cache_path, 'w') as f:
        json.dump(result, f)

    time.sleep(DELAY_SECONDS)
    return result

# Quick test on one film
test_row = sample.iloc[0]
test_result = extract_with_llm(test_row['plot'], test_row['union_id'])
print(f'Test film: {test_row["name"]} ({test_row["primary_genre"]})')
print(f'Error: {test_result["error"]}')
for cat in ['settings', 'characters', 'objects', 'events', 'themes']:
    print(f'  {cat}: {test_result[cat][:5]}')  # Show first 5

Test film: Antitrust (Action)
Error: None
  settings: ['software development company headquarters', 'portland oregon', 'secret database facility', 'public-access television station', 'corporate campus']
  characters: ['ambitious young programmer protagonist', 'ruthless tech ceo villain', 'girlfriend plant double agent', 'murdered best friend', 'co-worker reluctant ally']
  objects: ['stolen source code', 'surveillance dossiers', 'flagship software product', 'secret database', 'surveillance system']
  events: ['job recruitment by powerful corporation', 'relocation for new job', 'murder of best friend', 'discovery of corporate conspiracy', 'code theft from programmers worldwide']
  themes: ['corporate corruption', 'surveillance state', 'whistleblowing', 'betrayal', 'conspiracy']


In [ ]:
# Count how many are already cached
already_cached = sum(1 for _, row in sample.iterrows()
                     if os.path.exists(get_cache_path(row['union_id'])))
to_process = len(sample) - already_cached

print(f'Sample size:    {len(sample):,}')
print(f'Already cached: {already_cached:,}')
print(f'To process:     {to_process:,}')
if to_process > 0:
    est_minutes = to_process * DELAY_SECONDS / 60
    print(f'Estimated time: {est_minutes:.0f} minutes (at {DELAY_SECONDS}s per call)')

Sample size:    1,500
Already cached: 1,500
To process:     0


In [ ]:
# Run extraction on the full sample
llm_results = []
errors = 0

for _, row in tqdm(sample.iterrows(), total=len(sample), desc='LLM extraction'):
    result = extract_with_llm(row['plot'], row['union_id'])
    result['union_id'] = row['union_id']
    result['name'] = row['name']
    result['primary_genre'] = row['primary_genre']
    llm_results.append(result)
    if result['error']:
        errors += 1

print(f'\n✓ LLM extraction complete')
print(f'  Processed: {len(llm_results):,}')
print(f'  Errors:    {errors:,}')

LLM extraction:   0%|          | 0/1500 [00:00<?, ?it/s]


✓ LLM extraction complete
  Processed: 1,500
  Errors:    0


## 5. Compute Agreement Metrics

For each film in the sample, compare the spaCy-extracted noun elements (from Notebook 02) against the LLM-extracted elements (all categories flattened). We compute:

- **Precision**: Of spaCy's elements, what fraction also appear in the LLM output? (How much noise does spaCy have?)
- **Recall**: Of the LLM's elements, what fraction also appear in spaCy's output? (What does spaCy miss?)
- **Jaccard similarity**: Size of intersection / size of union

We use fuzzy matching: an element from one source matches the other if it appears as a substring. "sheriff" from spaCy matches "corrupt sheriff" from the LLM.

In [ ]:
def flatten_llm_elements(result):
    """Flatten all LLM-extracted elements into a single set of lowercase strings."""
    elements = set()
    for cat in ['settings', 'characters', 'objects', 'events', 'themes']:
        for elem in result.get(cat, []):
            if isinstance(elem, str):
                elements.add(elem.lower().strip())
    return elements

def fuzzy_element_in_set(element, target_set):
    """Check if element matches any item in target_set via substring matching.
    'sheriff' matches 'corrupt sheriff'; 'love' matches 'forbidden love'.
    """
    e = element.lower().strip()
    for t in target_set:
        if e in t or t in e:
            return True
    return False

def compute_agreement(spacy_elements, llm_elements):
    """Compute precision, recall, and Jaccard between two element sets."""
    if not spacy_elements and not llm_elements:
        return {'precision': 1.0, 'recall': 1.0, 'jaccard': 1.0,
                'n_spacy': 0, 'n_llm': 0, 'n_spacy_matched': 0, 'n_llm_matched': 0}

    spacy_set = set(spacy_elements)
    llm_set = llm_elements

    # Precision: of spaCy elements, how many match something in LLM?
    spacy_matched = sum(1 for s in spacy_set if fuzzy_element_in_set(s, llm_set))
    precision = spacy_matched / len(spacy_set) if spacy_set else 0

    # Recall: of LLM elements, how many match something in spaCy?
    llm_matched = sum(1 for l in llm_set if fuzzy_element_in_set(l, spacy_set))
    recall = llm_matched / len(llm_set) if llm_set else 0

    # Jaccard (using fuzzy matching for intersection)
    intersection = spacy_matched  # approximate
    union_size = len(spacy_set) + len(llm_set) - intersection
    jaccard = intersection / union_size if union_size > 0 else 0

    return {
        'precision': precision,
        'recall': recall,
        'jaccard': jaccard,
        'n_spacy': len(spacy_set),
        'n_llm': len(llm_set),
        'n_spacy_matched': spacy_matched,
        'n_llm_matched': llm_matched
    }

In [ ]:
# Build spaCy element lookups for the sample films.
#   spacy_by_type      : per element type (the fair, like-for-like comparison)
#   spacy_by_film      : pooled set across all types (kept for the exploratory
#                        "what did each side miss" cells further down)
#   spacy_chunks_by_film : chunks alone (kept for the chunk-specific cell)
SPACY_TYPES = ['noun', 'verb', 'chunk', 'entity']

spacy_by_type = {}
spacy_by_film = {}
spacy_chunks_by_film = {}
for _, row in elements.iterrows():
    uid = row['union_id']
    noun_set = set(row['nouns']) if isinstance(row['nouns'], list) else set()
    verb_set = set(row['verbs']) if isinstance(row['verbs'], list) else set()
    chunk_set = set(row['chunks']) if (HAS_CHUNKS and isinstance(row['chunks'], list)) else set()
    entity_set = set()
    if isinstance(row['entities'], list):
        for e in row['entities']:
            if isinstance(e, dict):
                entity_set.add(e.get('text', '').lower())
    spacy_by_type[uid] = {'noun': noun_set, 'verb': verb_set,
                          'chunk': chunk_set, 'entity': entity_set}
    spacy_by_film[uid] = noun_set | verb_set | chunk_set | entity_set
    spacy_chunks_by_film[uid] = chunk_set

# Per-film agreement: one pooled comparison plus one per element type.
# Pooled precision/recall/jaccard/n_spacy are kept on the frame so the
# downstream exploratory cells keep working; the per-type columns are the
# ones the report below actually uses.
agreement_rows = []
for result in llm_results:
    uid = result['union_id']
    if result['error']:
        continue
    llm_elems = flatten_llm_elements(result)

    row = {'union_id': uid, 'name': result['name'],
           'genre': result['primary_genre'], 'n_llm': len(llm_elems)}

    pooled = compute_agreement(spacy_by_film.get(uid, set()), llm_elems)
    row['precision'] = pooled['precision']   # pooled - kept for cells 25-27
    row['recall']    = pooled['recall']      # pooled - the meaningful recall
    row['jaccard']   = pooled['jaccard']
    row['n_spacy']   = pooled['n_spacy']

    for t in SPACY_TYPES:
        a = compute_agreement(spacy_by_type[uid][t], llm_elems)
        row[f'{t}_precision'] = a['precision']
        row[f'{t}_recall']    = a['recall']
        row[f'{t}_jaccard']   = a['jaccard']
        row[f'n_{t}']         = a['n_spacy']
    agreement_rows.append(row)

agreement_df = pd.DataFrame(agreement_rows)
print(f'Agreement computed for {len(agreement_df):,} films '
      f'(excluding {errors} errors)')

Agreement computed for 1,500 films (excluding 0 errors)


## 6. Agreement Report

In [ ]:
print('=' * 70)
print('AGREEMENT: spaCy vs. Claude Sonnet 4.6')
print('=' * 70)
print(f'\nSample size: {len(agreement_df):,} films')
print(f'Avg LLM elements per film: {agreement_df["n_llm"].mean():.1f}')

# --- per-type precision ---------------------------------------------------
# Of the spaCy elements of each type, what fraction did the LLM independently
# corroborate? Averaged only over films that actually have elements of that
# type, so films with zero chunks do not drag the chunk number down.
print('\nPER-TYPE PRECISION  (fraction of spaCy elements the LLM corroborated)')
for t in ['chunk', 'entity', 'noun', 'verb']:
    mask = agreement_df[f'n_{t}'] > 0
    p = agreement_df.loc[mask, f'{t}_precision']
    avg_n = agreement_df.loc[mask, f'n_{t}'].mean()
    if t == 'chunk':
        note = '   <- multi-word; closest to LLM output, the fair comparison'
    elif t == 'entity':
        note = '   <- named entities'
    else:
        note = '   <- single word; LLM not prompted for these, interpret with care'
    print(f'  {t:<8} mean={p.mean():.3f}  median={p.median():.3f}  '
          f'(avg {avg_n:.1f}/film, N={mask.sum()}){note}')

# --- pooled recall --------------------------------------------------------
# Of the LLM elements, what fraction did spaCy capture under ANY type? This
# is the meaningful cross-type question: does spaCy find what the LLM finds?
r = agreement_df['recall']
print('\nPOOLED RECALL  (fraction of LLM elements spaCy captured under any type)')
print(f'  mean={r.mean():.3f}  median={r.median():.3f}  std={r.std():.3f}')
print('  low recall is expected and informative: the LLM extracts abstract')
print('  themes (redemption, coming of age) that spaCy cannot reach.')

AGREEMENT: spaCy vs. Claude Sonnet 4.6

Sample size: 1,500 films
Avg LLM elements per film: 42.0

PER-TYPE PRECISION  (fraction of spaCy elements the LLM corroborated)
  chunk    mean=0.387  median=0.364  (avg 16.8/film, N=1496)   <- multi-word; closest to LLM output, the fair comparison
  entity   mean=0.432  median=0.429  (avg 5.8/film, N=1353)   <- named entities
  noun     mean=0.558  median=0.545  (avg 42.1/film, N=1500)   <- single word; LLM not prompted for these, interpret with care
  verb     mean=0.245  median=0.215  (avg 29.8/film, N=1499)   <- single word; LLM not prompted for these, interpret with care

POOLED RECALL  (fraction of LLM elements spaCy captured under any type)
  mean=0.657  median=0.674  std=0.143
  low recall is expected and informative: the LLM extracts abstract
  themes (redemption, coming of age) that spaCy cannot reach.


In [ ]:
# By genre: chunk precision (the fair like-for-like comparison) and pooled
# recall (LLM coverage). N is the per-genre film count - with the sampler
# fixed this should be balanced rather than one giant genre and the rest tiny.
print('\nBY GENRE:\n')
print(f'{"Genre":<18} {"ChunkPrec":>11} {"Recall":>10} {"N":>6}')
print('-' * 47)
for genre in CANONICAL_GENRES:
    gdf = agreement_df[agreement_df['genre'] == genre]
    if len(gdf) == 0:
        continue
    cmask = gdf['n_chunk'] > 0
    cp = gdf.loc[cmask, 'chunk_precision'].mean() if cmask.any() else float('nan')
    print(f'{genre:<18} {cp:>11.3f} {gdf["recall"].mean():>10.3f} {len(gdf):>6}')


BY GENRE:

Genre                ChunkPrec     Recall      N
-----------------------------------------------
Action                   0.380      0.683     89
Adventure                0.368      0.703     89
Animation                0.382      0.676     89
Comedy                   0.391      0.634     89
Crime                    0.394      0.627     88
Drama                    0.373      0.620     88
Family                   0.350      0.649     88
Fantasy                  0.380      0.686     88
History                  0.329      0.638     88
Horror                   0.439      0.694     88
Music                    0.369      0.627     88
Mystery                  0.449      0.630     88
Romance                  0.385      0.647     88
Science Fiction          0.374      0.727     88
Thriller                 0.377      0.646     88
War                      0.407      0.673     88
Western                  0.437      0.615     88


In [ ]:
# ---- Chunk-specific validation --------------------------------------------
# The overall metric folds chunks in with nouns, verbs, and entities. This
# isolates the multi-word chunk extraction so its quality is visible on its
# own: of the spaCy chunks in each film, how many does the LLM corroborate?
chunk_rows = []
for result in llm_results:
    if result['error']:
        continue
    uid = result['union_id']
    chunk_set = spacy_chunks_by_film.get(uid, set())
    if not chunk_set:
        continue
    llm_elems = flatten_llm_elements(result)
    matched = sum(1 for c in chunk_set if fuzzy_element_in_set(c, llm_elems))
    chunk_rows.append({
        'union_id': uid,
        'genre': result['primary_genre'],
        'n_chunks': len(chunk_set),
        'n_chunks_matched': matched,
        'chunk_precision': matched / len(chunk_set),
    })

chunk_df = pd.DataFrame(chunk_rows)
print('=' * 70)
print('CHUNK-SPECIFIC VALIDATION: multi-word spaCy elements vs. Claude')
print('=' * 70)
if len(chunk_df):
    print(f'\nFilms with at least one chunk: {len(chunk_df):,}')
    print(f'Mean chunks per film:          {chunk_df["n_chunks"].mean():.1f}')
    print(f'Mean chunk precision:          {chunk_df["chunk_precision"].mean():.3f}')
    print(f'  (fraction of spaCy chunks the LLM independently identified)')
    print(f'\nChunk precision by genre:')
    for genre in CANONICAL_GENRES:
        g = chunk_df[chunk_df['genre'] == genre]
        if len(g) == 0:
            continue
        print(f'  {genre:<18} {g["chunk_precision"].mean():.3f}  (N={len(g)})')

    # Concrete examples: chunks the LLM confirmed vs. chunks it did not
    print(f'\nExamples (matched = LLM also found it; unmatched = spaCy only):')
    shown = 0
    for result in llm_results:
        if result['error'] or shown >= 4:
            continue
        uid = result['union_id']
        chunk_set = spacy_chunks_by_film.get(uid, set())
        if len(chunk_set) < 4:
            continue
        llm_elems = flatten_llm_elements(result)
        matched = [c for c in chunk_set if fuzzy_element_in_set(c, llm_elems)]
        unmatched = [c for c in chunk_set if not fuzzy_element_in_set(c, llm_elems)]
        print(f'\n  {result["name"]} ({result["primary_genre"]})')
        print(f'    matched   : {matched[:6]}')
        print(f'    unmatched : {unmatched[:6]}')
        shown += 1
else:
    print('\nNo chunks found in the sample. Did Notebook 02 run with chunk '
          'extraction enabled? Check that elements_by_film.parquet has a '
          '"chunks" column.')

CHUNK-SPECIFIC VALIDATION: multi-word spaCy elements vs. Claude

Films with at least one chunk: 1,496
Mean chunks per film:          16.8
Mean chunk precision:          0.387
  (fraction of spaCy chunks the LLM independently identified)

Chunk precision by genre:
  Action             0.380  (N=88)
  Adventure          0.368  (N=89)
  Animation          0.382  (N=89)
  Comedy             0.391  (N=89)
  Crime              0.394  (N=88)
  Drama              0.373  (N=88)
  Family             0.350  (N=88)
  Fantasy            0.380  (N=88)
  History            0.329  (N=88)
  Horror             0.439  (N=88)
  Music              0.369  (N=87)
  Mystery            0.449  (N=87)
  Romance            0.385  (N=88)
  Science Fiction    0.374  (N=88)
  Thriller           0.377  (N=87)
  War                0.407  (N=88)
  Western            0.437  (N=88)

Examples (matched = LLM also found it; unmatched = spaCy only):

  Antitrust (Action)
    matched   : ['double agent', 'extensive surveillan

In [ ]:
# What does the LLM capture that spaCy misses? (Low recall = LLM has elements spaCy doesn't)
# Sample some films with low recall to see what the LLM found
print('\nEXAMPLES: Elements the LLM captured that spaCy missed')
print('(Films with the lowest recall — largest gap between methods)\n')

low_recall = agreement_df.nsmallest(10, 'recall')

for _, row in low_recall.head(5).iterrows():
    uid = row['union_id']
    llm_result = next(r for r in llm_results if r['union_id'] == uid)
    spacy_elems = spacy_by_film.get(uid, set())
    llm_elems = flatten_llm_elements(llm_result)

    # Elements in LLM but not in spaCy
    llm_only = [e for e in llm_elems if not fuzzy_element_in_set(e, spacy_elems)]

    print(f'--- {row["name"]} ({row["genre"]}) ---')
    print(f'  spaCy elements: {row["n_spacy"]}, LLM elements: {row["n_llm"]}')
    print(f'  Precision: {row["precision"]:.2f}, Recall: {row["recall"]:.2f}')
    print(f'  LLM-only elements: {llm_only[:10]}')
    print()


EXAMPLES: Elements the LLM captured that spaCy missed
(Films with the lowest recall — largest gap between methods)

--- Shakedown (Mystery) ---
  spaCy elements: 22, LLM elements: 22
  Precision: 0.14, Recall: 0.14
  LLM-only elements: ['city streets', 'double-cross', 'greed', 'unscrupulous photographer', 'self-preservation', 'posthumous justice', 'betrayal', 'enforcer', 'framing of henchman', 'henchman']

--- Hunt Angels (Drama) ---
  spaCy elements: 18, LLM elements: 34
  Precision: 0.39, Recall: 0.15
  LLM-only elements: ['cultural identity', 'passionate female partner', 'rural australia', 'film camera', 'law enforcement pursuers', 'confrontation with corrupt officials', 'production gear', 'film equipment', 'prolonged chase across vast distances', 'pursuing dreams against all odds']

--- Masterson of Kansas (Western) ---
  spaCy elements: 13, LLM elements: 30
  Precision: 0.23, Recall: 0.17
  LLM-only elements: ['open range', 'frontier doctor', 'good vs evil', 'greed', 'honest ranc

In [ ]:
# What does spaCy capture that the LLM misses? (Low precision = spaCy has noise the LLM doesn't)
print('EXAMPLES: Elements spaCy extracted that the LLM did NOT identify')
print('(Films with the lowest precision — potential spaCy noise)\n')

low_precision = agreement_df.nsmallest(10, 'precision')

for _, row in low_precision.head(5).iterrows():
    uid = row['union_id']
    llm_result = next(r for r in llm_results if r['union_id'] == uid)
    spacy_elems = spacy_by_film.get(uid, set())
    llm_elems = flatten_llm_elements(llm_result)

    # Elements in spaCy but not in LLM
    spacy_only = [e for e in spacy_elems if not fuzzy_element_in_set(e, llm_elems)]

    print(f'--- {row["name"]} ({row["genre"]}) ---')
    print(f'  spaCy elements: {row["n_spacy"]}, LLM elements: {row["n_llm"]}')
    print(f'  Precision: {row["precision"]:.2f}, Recall: {row["recall"]:.2f}')
    print(f'  spaCy-only elements: {spacy_only[:10]}')
    print()

EXAMPLES: Elements spaCy extracted that the LLM did NOT identify
(Films with the lowest precision — potential spaCy noise)

--- Grand Exit (Mystery) ---
  spaCy elements: 553, LLM elements: 48
  Precision: 0.10, Recall: 0.85
  spaCy-only elements: ['outlook', 'disbelieving fire chief', 'unison', 'order', 'seller', 'troy', 'fletcher apartment', 'fire insurance company', 'admit', 'fire engine']

--- Alley Cats Strike (Family) ---
  spaCy elements: 546, LLM elements: 52
  Precision: 0.13, Recall: 0.88
  spaCy-only elements: ['release', 'non swing music', 'name', 'order', 'comfort', 'smile', 'admit', 'heart attack', 'unsubstitute', 'book']

--- Shakedown (Mystery) ---
  spaCy elements: 22, LLM elements: 22
  Precision: 0.14, Recall: 0.14
  spaCy-only elements: ['liking', 'frame', 'capture', 'like', 'celluloid', 'event', 'send', 'colton', 'palmer', 'inform']

--- The Last Dragon (Fantasy) ---
  spaCy elements: 539, LLM elements: 49
  Precision: 0.16, Recall: 0.96
  spaCy-only elements: ['ga

In [ ]:
# Aggregate: what element categories does the LLM find most that spaCy misses?
print('LLM ELEMENT CATEGORIES — what does the LLM extract that spaCy cannot?\n')

category_counts = Counter()
category_examples = {cat: Counter() for cat in ['settings', 'characters', 'objects', 'events', 'themes']}

for result in llm_results:
    if result['error']:
        continue
    uid = result['union_id']
    spacy_elems = spacy_by_film.get(uid, set())

    for cat in ['settings', 'characters', 'objects', 'events', 'themes']:
        for elem in result.get(cat, []):
            if isinstance(elem, str):
                elem_lower = elem.lower().strip()
                if not fuzzy_element_in_set(elem_lower, spacy_elems):
                    category_counts[cat] += 1
                    category_examples[cat][elem_lower] += 1

print(f'{"Category":<15} {"LLM-only elements":>20}')
print('-' * 38)
for cat in ['settings', 'characters', 'objects', 'events', 'themes']:
    print(f'{cat:<15} {category_counts[cat]:>20,}')

print('\nMost common LLM-only elements by category:\n')
for cat in ['settings', 'characters', 'objects', 'events', 'themes']:
    top = category_examples[cat].most_common(10)
    top_str = ', '.join([f'{e} ({c})' for e, c in top])
    print(f'  {cat.upper()}: {top_str}')

LLM ELEMENT CATEGORIES — what does the LLM extract that spaCy cannot?

Category           LLM-only elements
--------------------------------------
settings                       1,815
characters                     3,316
objects                        1,153
events                         3,285
themes                        10,337

Most common LLM-only elements by category:

  SETTINGS: urban environment (44), family home (35), courtroom (25), frontier town (23), city streets (18), suburban neighborhood (17), small town (16), domestic interior (14), television studio (13), domestic household (12)
  CHARACTERS: mentor figure (38), love interest (18), romantic rival (16), romantic lead (15), femme fatale (13), reluctant hero (13), grieving widower (9), damsel in distress (8), child protagonist (8), comic sidekick (8)
  OBJECTS: gun (44), camera (19), military uniform (18), musical instruments (18), microphone (17), military weapons (16), weapons (15), guns (10), firearms (10), legal docum

## 7. Save Validation Report

In [25]:
# Save agreement DataFrame
agreement_path = f'{OUTPUT_DIR}/agreement_metrics.parquet'
agreement_df.to_parquet(agreement_path, engine='pyarrow')
print(f'Saved: {agreement_path}')

# Save full LLM results.
# Some columns ('characters', 'settings', 'events', 'themes', 'objects') hold
# python lists which pyarrow cannot serialise reliably from an inferred object
# column: it sometimes infers binary type and rejects the list. Serialise list/
# dict columns as JSON strings before save; downstream consumers (which are
# rare for this artifact; it's a debug aid) can json.loads back to lists.
import json as _json
llm_df = pd.DataFrame(llm_results)
_list_cols = [c for c in llm_df.columns
              if llm_df[c].apply(lambda v: isinstance(v, (list, dict))).any()]
for c in _list_cols:
    llm_df[c] = llm_df[c].apply(
        lambda v: _json.dumps(v) if isinstance(v, (list, dict)) else v)
llm_path = f'{OUTPUT_DIR}/llm_extractions.parquet'
llm_df.to_parquet(llm_path, engine='pyarrow')
print(f'Saved: {llm_path}  (list/dict columns serialised as JSON: {_list_cols})')

# Write text report
report_path = f'{OUTPUT_DIR}/validation_report.txt'
with open(report_path, 'w') as f:
    f.write('ELEMENT EXTRACTION VALIDATION REPORT\n')
    f.write('=' * 60 + '\n')
    f.write(f'spaCy model: en_core_web_lg\n')
    f.write(f'LLM model: Claude Sonnet 4.6 ({MODEL})\n')
    f.write(f'Sample size: {len(agreement_df)} films '
            f'(stratified by genre, mode={STRATIFICATION})\n\n')

    f.write('PER-TYPE PRECISION (fraction of spaCy elements the LLM corroborated)\n')
    for t in ['chunk', 'entity', 'noun', 'verb']:
        mask = agreement_df[f'n_{t}'] > 0
        p = agreement_df.loc[mask, f'{t}_precision']
        f.write(f'  {t:<8} mean={p.mean():.3f}  median={p.median():.3f}  '
                f'N={mask.sum()}\n')
    f.write('  (chunk = fair comparison; noun/verb = LLM not prompted for these)\n')

    r = agreement_df['recall']
    f.write(f'\nPOOLED RECALL (LLM elements spaCy captured under any type)\n')
    f.write(f'  mean={r.mean():.3f}, median={r.median():.3f}, std={r.std():.3f}\n')

    f.write(f'\nAvg spaCy elements/film (pooled): {agreement_df["n_spacy"].mean():.1f}\n')
    f.write(f'Avg LLM elements/film:           {agreement_df["n_llm"].mean():.1f}\n\n')

    f.write('BY GENRE (chunk precision, pooled recall, N)\n')
    for genre in CANONICAL_GENRES:
        gdf = agreement_df[agreement_df['genre'] == genre]
        if len(gdf) == 0:
            continue
        cmask = gdf['n_chunk'] > 0
        cp = gdf.loc[cmask, 'chunk_precision'].mean() if cmask.any() else float('nan')
        f.write(f'  {genre:<18} chunkP={cp:.3f}  R={gdf["recall"].mean():.3f}  '
                f'N={len(gdf)}\n')

    f.write(f'\nLLM-ONLY ELEMENTS BY CATEGORY\n')
    for cat in ['settings', 'characters', 'objects', 'events', 'themes']:
        f.write(f'  {cat}: {category_counts[cat]:,} elements\n')
        top = category_examples[cat].most_common(10)
        for elem, count in top:
            f.write(f'    {elem} ({count})\n')

    f.write(f'\nCHUNK-SPECIFIC VALIDATION (multi-word elements)\n')
    if len(chunk_df):
        f.write(f'  films with >=1 chunk: {len(chunk_df)}\n')
        f.write(f'  mean chunks per film: {chunk_df["n_chunks"].mean():.1f}\n')
        f.write(f'  mean chunk precision: {chunk_df["chunk_precision"].mean():.3f}\n')
        for genre in CANONICAL_GENRES:
            g = chunk_df[chunk_df['genre'] == genre]
            if len(g) == 0:
                continue
            f.write(f'    {genre:<18} {g["chunk_precision"].mean():.3f}  N={len(g)}\n')
    else:
        f.write('  no chunks found in sample (re-run Notebook 02 with chunk extraction)\n')

# Save chunk-specific agreement frame
if len(chunk_df):
    chunk_path = f'{OUTPUT_DIR}/chunk_agreement.parquet'
    chunk_df.to_parquet(chunk_path, engine='pyarrow')
    print(f'Saved: {chunk_path}')

print(f'Saved: {report_path}')

Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02b_llm_validation/agreement_metrics.parquet
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02b_llm_validation/llm_extractions.parquet  (list/dict columns serialised as JSON: ['settings', 'characters', 'objects', 'events', 'themes'])
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02b_llm_validation/chunk_agreement.parquet
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02b_llm_validation/validation_report.txt


## 8. Interpretation

**Expected findings:**

- **Precision** (spaCy elements confirmed by LLM) should be moderate to high (0.4–0.7). When spaCy identifies a noun as a narrative element, the LLM usually agrees it's relevant, though spaCy also captures some noise words the LLM ignores.

- **Recall** (LLM elements confirmed by spaCy) should be moderate (0.3–0.5). The LLM captures abstract tropes and multi-word expressions that spaCy fundamentally cannot identify: "redemption arc," "chosen one," "forbidden love," "car chase," "final showdown."

- **Themes and events** should show the largest gap: these are the categories where semantic understanding (LLM) diverges most from syntactic parsing (spaCy).

- **Settings and objects** should show the smallest gap: these are concrete, noun-heavy categories where spaCy performs well.

**What this means for the paper:**

The validation establishes that spaCy captures the concrete, surface-level elements that drive the stockiness computation (settings, objects, character types) with reasonable fidelity. The gap is in abstract narrative tropes, which are important for a full understanding of genre conventions but are not what the current stockiness measures are designed to capture. A future extension could incorporate LLM-based extraction for the full corpus, but for the current paper, the spaCy pipeline captures enough genre signal to produce valid stockiness measures — as confirmed by the top-element sanity checks in Notebook 02 and the agreement metrics here.

**Next: Notebook 03** computes stockiness measures on the element index from Notebook 02.